# Qwen3.5-9B · 첫 실험: 추론 해상도 768²

원본 baseline의 **설정 → 데이터 → 모델 → Dataset/Collator → DataLoader → 학습/저장 → 추론/CSV** 순서를 유지합니다.
패키지 설치·가상환경 생성·별도 worker 실행은 없습니다. Qwen3.5가 이미 실행되는 커널에서 **Restart Kernel → Run All** 하세요.

- 공통: NF4 4비트, BF16, 학습180개×1epoch, 학습384², lr1e-4, LoRA r8/alpha16/dropout.05.
- 이번 파일: 검증/test **추론만 768²**. 검증500개, 최종 확인용500개는 별도 보관하고 이번에는 평가하지 않습니다.
- **PC1(384)**: 기존 완료 실행 폴더가 있으면 LoRA를 재사용. 없으면 384²로 한 번 학습하여 공통 폴더 생성.
- **PC2~4**: PC1의 `output/qwen35_resolution_common_v1/` 전체를 같은 상대 경로로 복사한 뒤 Run All.
  네 PC가 같은 어댑터와 분할을 사용하는 데 필요한 복사이며 환경 설치가 아닙니다. 원본9B 모델 파일은 각 PC가 자동 다운로드합니다.
- 기본값은 검증 후 전체 test 추론 및 `submission.csv` 생성까지 수행합니다. `RUN_TEST=False`이면 검증까지만 실행합니다.
- 신규 GPU 실행·점수는 미검증. 고해상도 OOM/길이 초과 시 자동 축소하지 않습니다.


## 1. 경로·설정


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, time, uuid, shutil

ASSET_DIR = Path("downloads")
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
MODEL_REPO = "Qwen/Qwen3.5-9B"
MODEL_REVISION = "c202236235762e1c871ad0ccb60c8ee5ba337b9a"
# 이전 Qwen RunAll과 동일한 다운로드 위치라 완료 파일을 재사용할 수 있습니다.
MODEL_DIR = ASSET_DIR / "models" / "Qwen3_5_9B" / MODEL_REVISION
COMMON_DIR = OUTPUT_DIR / "qwen35_resolution_common_v1"
SAVE_DIR = COMMON_DIR / "adapter"

PC_NUMBER = 4
IMAGE_SIZE = 768                     # 이번 실험의 추론 픽셀 예산
TRAIN_IMAGE_SIZE = 384                   # 4개 파일 모두 같은 학습 해상도
TRAIN_N, VALID_N, HOLDOUT_N = 180, 500, 500
SEED = 42
EPOCHS, GRAD_ACCUM, LEARNING_RATE = 1, 4, 1e-4
MAX_NEW_TOKENS, MAX_INPUT_TOKENS = 2, 4096
RUN_TEST = True
# 사용자 제공 실행 출력에 나타난 완료 폴더. 존재하면 180개 학습 결과를 재사용합니다.
# 다른 완료 폴더를 쓰려면 같은 형식의 run_config.json/split_manifest.csv가 있는 경로로 변경하세요.
SOURCE_RUN_DIR = Path("output/TASK-004/EXP-011/20260921_052420_1293e54e")
EXPERIMENT_ID = f"RES1-PC{PC_NUMBER}-{IMAGE_SIZE}"
RUN_DIR = OUTPUT_DIR / "qwen35_resolution_v1" / EXPERIMENT_ID / (
    time.strftime("%Y%m%d_%H%M%S", time.gmtime()) + "_" + uuid.uuid4().hex[:6])

for filename in ("train.csv", "test.csv", "sample_submission.csv"):
    if not (DATA_DIR / filename).is_file():
        raise FileNotFoundError(f"DATA_DIR 확인: {DATA_DIR / filename}")
if PC_NUMBER != 1 and not (COMMON_DIR / "bundle.json").is_file():
    raise FileNotFoundError("PC1에서 생성한 output/qwen35_resolution_common_v1 폴더 전체를 먼저 복사하세요.")
RUN_DIR.mkdir(parents=True, exist_ok=False)

def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# 재실행 시에도 다운로드 단계만 인터넷 허용. 학습/추론은 아래 단계에서 차단합니다.
if not hasattr(sys, "_qwen35_resolution_network"):
    sys._qwen35_resolution_network = {"download": True}
    def network_guard(event, args):
        if sys._qwen35_resolution_network["download"]:
            return
        if event == "socket.connect":
            addr = args[1]
        elif event == "socket.getaddrinfo":
            addr = (args[0],)
        elif event == "socket.sendto":
            addr = args[-1]
        else:
            return
        host = addr[0] if isinstance(addr, tuple) else addr
        # Jupyter의 로컬 통신은 유지하고 외부 연결만 막습니다.
        if host not in ("127.0.0.1", "::1", "localhost", b"localhost"):
            raise RuntimeError("모델 준비 이후 외부 네트워크 연결은 차단됩니다.")
    sys.addaudithook(network_guard)
sys._qwen35_resolution_network["download"] = True
print("결과 폴더:", RUN_DIR)


In [ ]:
# 현재 커널을 그대로 사용합니다. 설치/업데이트 명령은 없습니다.
import re, math, random, gc, csv
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from dataclasses import dataclass
from typing import Any
from torch.utils.data import Dataset, DataLoader
import torch
import transformers, peft
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from transformers import get_linear_schedule_with_warmup
from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
from peft import LoraConfig, get_peft_model, PeftModel
from tqdm.auto import tqdm
from importlib.metadata import version

if "qwen3_5" not in CONFIG_MAPPING_NAMES:
    raise RuntimeError("현재 커널은 Qwen3.5를 지원하지 않습니다. 기존 Qwen3.5 실행에 성공한 커널을 선택하세요. 자동 설치는 하지 않습니다.")
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError("BF16 지원 CUDA GPU가 필요합니다.")
device, DTYPE = "cuda:0", torch.bfloat16
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)
ENVIRONMENT = {"python":sys.version, "gpu":torch.cuda.get_device_name(0),
    "vram_gib":torch.cuda.get_device_properties(0).total_memory / 2**30,
    "packages":{x:version(x) for x in ("torch", "transformers", "peft", "bitsandbytes", "accelerate", "pandas", "pillow")}}
write_json(RUN_DIR / "environment.json", ENVIRONMENT)
print(ENVIRONMENT)


In [ ]:
# 모델 파일만 자동 다운로드. 외부 추론 API를 사용하지 않습니다.
import urllib.request
import urllib.parse
MODEL_FILES = ["chat_template.jinja", "config.json", "merges.txt",
    "model.safetensors-00001-of-00004.safetensors", "model.safetensors-00002-of-00004.safetensors",
    "model.safetensors-00003-of-00004.safetensors", "model.safetensors-00004-of-00004.safetensors",
    "model.safetensors.index.json", "preprocessor_config.json", "tokenizer.json",
    "tokenizer_config.json", "video_preprocessor_config.json", "vocab.json"]
MODEL_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = MODEL_DIR / "download_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
files = manifest.get("files", {}) if manifest.get("revision") == MODEL_REVISION else {}

class HTTPSRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        if urllib.parse.urlparse(newurl).scheme != "https":
            raise RuntimeError("HTTPS 모델 다운로드만 허용합니다.")
        return super().redirect_request(req, fp, code, msg, headers, newurl)
opener = urllib.request.build_opener(HTTPSRedirect())
try:
    for name in MODEL_FILES:
        path = MODEL_DIR / name
        old = files.get(name, {})
        if path.is_file() and path.stat().st_size == old.get("size") and file_hash(path) == old.get("sha256"):
            print("재사용:", name)
            continue
        url = f"https://huggingface.co/{MODEL_REPO}/resolve/{MODEL_REVISION}/{name}"
        partial = path.with_name(path.name + ".part")
        for attempt in range(3):
            try:
                request = urllib.request.Request(url, headers={"User-Agent":"local-qwen35-download/1.0"})
                digest, size = hashlib.sha256(), 0
                with opener.open(request, timeout=120) as response, open(partial, "wb") as out:
                    if "text/html" in response.headers.get("Content-Type", ""):
                        raise RuntimeError("모델 대신 HTML 응답을 받았습니다.")
                    expected = response.headers.get("Content-Length")
                    with tqdm(total=int(expected) if expected else None, unit="B", unit_scale=True, desc=name) as bar:
                        for chunk in iter(lambda:response.read(4*1024*1024), b""):
                            out.write(chunk); digest.update(chunk); size += len(chunk); bar.update(len(chunk))
                if not size or (expected and size != int(expected)):
                    raise RuntimeError("다운로드 파일 크기 불일치")
                partial.replace(path)
                files[name] = {"size":size, "sha256":digest.hexdigest()}
                write_json(manifest_path, {"model_id":MODEL_REPO, "revision":MODEL_REVISION, "files":files})
                break
            except Exception:
                if attempt == 2:
                    raise
finally:
    sys._qwen35_resolution_network["download"] = False
    os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_DATASETS_OFFLINE="1",
        HF_HUB_DISABLE_TELEMETRY="1", WANDB_DISABLED="true", TOKENIZERS_PARALLELISM="false")
shutil.copy2(manifest_path, RUN_DIR / "model_assets.json")
print("모델 준비 완료. 이후 로컬 파일만 사용합니다.")


## 2. 데이터·고정 분할
학습은 원본의 seed42/200개 추출→앞180개를 유지합니다. 검증은 학습 이미지와 분리한 500개로 확장합니다.
PC1에서 픽셀 해시로 동일 이미지를 묶어 검증/최종확인 분할을 만들고, 나머지 PC는 저장된 분할을 그대로 사용합니다.
유사 장면·유사 이미지 검토는 별도이며, 정확히 동일한 이미지 검사만으로 모든 누수를 배제하지는 못합니다.


In [ ]:
def read_table(path, required):
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    if not set(required) <= set(df.columns) or df.empty:
        raise ValueError(f"컬럼/데이터 확인: {path}")
    if df.id.duplicated().any() or any(df[c].str.strip().eq("").any() for c in required):
        raise ValueError(f"빈 필수 값 또는 중복 ID: {path}")
    return df

def image_path(value):
    value = str(value).replace("\\", "/")
    root = DATA_DIR.resolve(); path = (root / value).resolve()
    if ":" in value or not path.is_relative_to(root) or not path.is_file():
        raise ValueError(f"DATA_DIR 내부 이미지 경로 확인: {value}")
    return path

cols = ["id", "path", "question", "a", "b", "c", "d"]
train_df = read_table(DATA_DIR / "train.csv", cols + ["answer"])
test_df = read_table(DATA_DIR / "test.csv", cols)
sample_submission = read_table(DATA_DIR / "sample_submission.csv", ["id"])
if not train_df.answer.isin(list("abcd")).all():
    raise ValueError("train answer는 a~d여야 합니다.")
if set(sample_submission.columns) != {"id", "answer"} or set(sample_submission.id) != set(test_df.id):
    raise ValueError("제출 형식/ID 불일치")
if set(train_df.id) & set(test_df.id):
    raise ValueError("train/test ID 중복")
TRAIN_CSV_HASH = file_hash(DATA_DIR / "train.csv")
SOURCE_ADAPTER = None
COMMON_DIR.mkdir(parents=True, exist_ok=True)

if (COMMON_DIR / "bundle.json").exists():
    bundle = json.loads((COMMON_DIR / "bundle.json").read_text(encoding="utf-8"))
    if (bundle["train_csv_sha256"] != TRAIN_CSV_HASH or bundle["model_revision"] != MODEL_REVISION
        or bundle["model_id"] != MODEL_REPO or bundle["training_image_size"] != TRAIN_IMAGE_SIZE
        or bundle["training_n"] != TRAIN_N):
        raise ValueError("공통 폴더와 데이터/모델 버전이 다릅니다.")
    if bundle["packages"] != ENVIRONMENT["packages"]:
        raise ValueError("PC1과 패키지 버전이 다릅니다. 동일한 기존 실행 환경을 사용하세요.")
    for relative, digest in bundle["files"].items():
        if file_hash(COMMON_DIR / relative) != digest:
            raise ValueError(f"공통 파일 해시 불일치: {relative}")
    if bundle["model_files"] != files:
        raise ValueError("PC1과 원본 모델 파일 해시가 다릅니다.")
    split_df = read_table(COMMON_DIR / "split_manifest.csv", ["id", "split"])
else:
    if PC_NUMBER != 1:
        raise RuntimeError("PC1의 완성된 공통 폴더가 필요합니다.")
    selected = train_df.sample(n=200, random_state=SEED).reset_index(drop=True)
    training_ids = selected.id.iloc[:TRAIN_N].tolist()
    # 완료 실행을 재사용할 때 학습 ID와 핵심 조건을 먼저 확인합니다.
    if SOURCE_RUN_DIR.exists():
        source_cfg = json.loads((SOURCE_RUN_DIR / "run_config.json").read_text(encoding="utf-8"))
        source_split = read_table(SOURCE_RUN_DIR / "split_manifest.csv", ["id", "split"])
        source_data = json.loads((SOURCE_RUN_DIR / "data_manifest.json").read_text(encoding="utf-8"))
        source_metrics = json.loads((SOURCE_RUN_DIR / "train_metrics.json").read_text(encoding="utf-8"))
        expected = {"model_id":MODEL_REPO, "revision":MODEL_REVISION, "epochs":1,
                    "seed":42, "learning_rate":1e-4, "gradient_accumulation":4}
        if any(source_cfg.get(k) != v for k,v in expected.items()):
            raise ValueError("기존 실행의 모델/학습 조건이 이번 기준과 다릅니다.")
        if source_data["train_csv_sha256"] != TRAIN_CSV_HASH or source_metrics["epochs"] != 1:
            raise ValueError("기존 학습 데이터 또는 epoch 불일치")
        if source_split.loc[source_split.split.eq("train"), "id"].tolist() != training_ids:
            raise ValueError("기존 어댑터의 학습 ID가 baseline 180개와 다릅니다.")
        source_processor = json.loads((SOURCE_RUN_DIR / "processor_settings.json").read_text(encoding="utf-8"))
        source_lora = json.loads((SOURCE_RUN_DIR / "adapter_epoch1" / "adapter_config.json").read_text(encoding="utf-8"))
        if source_cfg.get("quantization") != "NF4/double-quant/BF16-compute":
            raise ValueError("기존 실행의 양자화 조건 불일치")
        if source_cfg.get("loss") != "full_text; padding/media/role-special tokens masked; EOS supervised":
            raise ValueError("기존 실행의 loss 정책 불일치")
        if source_processor["image_processor"].get("size") != {"shortest_edge":384**2, "longest_edge":384**2}:
            raise ValueError("기존 학습 해상도가 384²가 아닙니다.")
        if any(source_lora.get(k) != v for k,v in {"r":8,"lora_alpha":16,"lora_dropout":0.05,"bias":"none"}.items()):
            raise ValueError("기존 LoRA 설정 불일치")
        SOURCE_ADAPTER = SOURCE_RUN_DIR / "adapter_epoch1"
        if not (SOURCE_ADAPTER / "adapter_model.safetensors").is_file():
            raise FileNotFoundError(SOURCE_ADAPTER)
        print("기존 완료 LoRA 재사용:", SOURCE_ADAPTER)
    # 동일 픽셀 이미지 그룹을 train/valid/holdout 사이에서 분리합니다.
    image_rows, pixel_cache = [], {}
    for row in tqdm(train_df.to_dict("records"), desc="이미지 분할 점검"):
        path = image_path(row["path"])
        key = str(path)
        if key not in pixel_cache:
            with Image.open(path) as image:
                image = ImageOps.exif_transpose(image).convert("RGB")
                pixel_cache[key] = hashlib.sha256(str(image.size).encode()+image.tobytes()).hexdigest()
        image_rows.append({"id":row["id"], "pixel_sha256":pixel_cache[key]})
    audit = pd.DataFrame(image_rows)
    annotated = train_df.merge(audit, on="id", validate="one_to_one")
    excluded = set(annotated.loc[annotated.id.isin(training_ids), "pixel_sha256"])
    pool = annotated.loc[~annotated.pixel_sha256.isin(excluded)]
    groups = [g.id.tolist() for _, g in pool.groupby("pixel_sha256", sort=True)]
    random.Random(SEED).shuffle(groups)
    def take_groups(groups, n):
        ids, remaining = [], []
        for group in groups:
            if len(ids) + len(group) <= n:
                ids.extend(group)
            else:
                remaining.append(group)
        if len(ids) != n:
            raise ValueError(f"이미지 그룹을 유지하며 {n}개를 확보하지 못했습니다.")
        return ids, remaining
    valid_ids, remaining = take_groups(groups, VALID_N)
    holdout_ids, _ = take_groups(remaining, HOLDOUT_N)
    split_df = pd.DataFrame({"id":training_ids+valid_ids+holdout_ids,
        "split":["train"]*TRAIN_N+["valid"]*VALID_N+["holdout"]*HOLDOUT_N})
    split_df.to_csv(COMMON_DIR / "split_manifest.csv", index=False)
    audit.to_csv(COMMON_DIR / "image_audit.csv", index=False)
    bundle = None

if split_df.id.duplicated().any() or not set(split_df.id) <= set(train_df.id):
    raise ValueError("공통 분할 ID 불일치")
counts = split_df.split.value_counts().to_dict()
if counts != {"train":TRAIN_N, "valid":VALID_N, "holdout":HOLDOUT_N}:
    raise ValueError(f"분할 개수 불일치: {counts}")
indexed = train_df.set_index("id", drop=False)
train_subset = indexed.loc[split_df.loc[split_df.split.eq("train"), "id"]].reset_index(drop=True)
valid_subset = indexed.loc[split_df.loc[split_df.split.eq("valid"), "id"]].reset_index(drop=True)
shutil.copy2(COMMON_DIR / "split_manifest.csv", RUN_DIR / "split_manifest.csv")
print("학습", len(train_subset), "/ 검증", len(valid_subset), "/ 최종 확인용", HOLDOUT_N, "(미평가)")


## 3. 모델·Processor
NF4와 BF16을 사용합니다. 학습 시에는 384², 평가 직전에만 파일별 해상도로 변경합니다.
큰 임베딩/출력층을 일괄 FP32로 변환하는 k-bit 준비 대신, 기존 Qwen 성공 코드의 동결·norm FP32·checkpointing 처리를 적용합니다.


In [ ]:
MODEL_ID = str(MODEL_DIR)
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE,
    llm_int8_skip_modules=["visual", "vision_tower", "vision_model", "vpm",
                          "resampler", "multi_modal_projector", "mlp1", "lm_head"])
processor = AutoProcessor.from_pretrained(MODEL_ID, local_files_only=True)

def set_image_size(size):
    ip = processor.image_processor
    ip.size = {"shortest_edge":size*size, "longest_edge":size*size}
    if hasattr(ip, "min_pixels"): ip.min_pixels = size*size
    if hasattr(ip, "max_pixels"): ip.max_pixels = size*size

set_image_size(TRAIN_IMAGE_SIZE)
base_model = AutoModelForImageTextToText.from_pretrained(MODEL_ID,
    quantization_config=bnb_config, torch_dtype=DTYPE,
    device_map={"":"cuda:0"}, attn_implementation="sdpa", local_files_only=True)
base_model.config.use_cache = False
# 학습/재사용 경로 모두 norm 정밀도를 같게 유지합니다.
for name, p in base_model.named_parameters():
    p.requires_grad_(False)
    if "norm" in name.lower() and p.is_floating_point() and p.ndim == 1:
        p.data = p.data.to(torch.float32)

if bundle is not None or SOURCE_ADAPTER is not None:
    model = base_model       # 뒤 셀에서 저장된 어댑터를 로드합니다.
else:
    base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})
    base_model.enable_input_require_grads()
    suffixes = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
    excluded = {"visual", "vision_tower", "vision_model", "vpm", "resampler", "multi_modal_projector", "mlp1"}
    targets = [name for name, module in base_model.named_modules()
        if name.split(".")[-1] in suffixes and not set(name.split(".")) & excluded
        and hasattr(module, "weight")]
    if not targets: raise RuntimeError("LoRA 대상 모듈 없음")
    lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
        target_modules=targets, task_type="CAUSAL_LM")
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()


## 4. 프롬프트 템플릿


In [ ]:
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)
def build_mc_prompt(question, a, b, c, d):
    return (f"{question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
            "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.")


## 5. Custom Dataset·Collator
원본 클래스 이름과 흐름을 유지합니다. 입력 텍스트도 감독하는 기존 loss이며, padding/미디어/특수 토큰(EOS 제외)을 마스킹합니다.
검증 생성에는 assistant 정답 메시지를 추가하지 않습니다.


In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        with Image.open(image_path(row["path"])) as f:
            img = ImageOps.exif_transpose(f).convert("RGB")
        user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])
        messages = [
            {"role":"system", "content":[{"type":"text", "text":SYSTEM_INSTRUCT}]},
            {"role":"user", "content":[{"type":"image"}, {"type":"text", "text":user_text}]}
        ]
        if self.train:
            messages.append({"role":"assistant", "content":[{"type":"text", "text":row["answer"]}]})
        return {"messages":messages, "image":img}

@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts = [self.processor.apply_chat_template(sample["messages"], tokenize=False,
            add_generation_prompt=not self.train, enable_thinking=False) for sample in batch]
        enc = self.processor(text=texts, images=[x["image"] for x in batch],
            padding=True, return_tensors="pt", add_special_tokens=False)
        if enc["input_ids"].shape[1] > MAX_INPUT_TOKENS:
            raise ValueError("입력 길이 초과. 해상도나 상한을 자동 변경하지 않습니다.")
        if self.train:
            labels = enc["input_ids"].clone()
            labels[enc["attention_mask"] == 0] = -100
            for sid in self.processor.tokenizer.all_special_ids:
                if sid != self.processor.tokenizer.eos_token_id:
                    labels[enc["input_ids"] == sid] = -100
            enc["labels"] = labels
        return enc

def to_device(batch):
    return {k:v.to(device=device, dtype=DTYPE if v.is_floating_point() else v.dtype)
            for k,v in batch.items()}


## 6. DataLoader


In [ ]:
train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=False)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    collate_fn=DataCollator(processor, True), num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False,
    collate_fn=DataCollator(processor, False), num_workers=0)


## 7. Fine-tuning·공통 체크포인트
PC1만 필요한 경우 한 번 학습합니다. 이미 완성된 공통 폴더가 있으면 모든 PC에서 학습을 건너뜁니다.
저장 후 다시 로드한 동일 LoRA로 평가하며, 네 파일에서 학습 해상도는 항상 384²입니다.


In [ ]:
if bundle is None:
    if SOURCE_ADAPTER is not None:
        shutil.copytree(SOURCE_ADAPTER, SAVE_DIR, dirs_exist_ok=True)
        training = {"state":"reused_existing", "source":str(SOURCE_RUN_DIR),
                    "source_train_metrics":source_metrics}
    else:
        params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(params, lr=LEARNING_RATE, weight_decay=0.01)
        updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
        num_training_steps = EPOCHS * updates_per_epoch
        scheduler = get_linear_schedule_with_warmup(optimizer, int(num_training_steps*0.03), num_training_steps)
        optimizer.zero_grad(set_to_none=True)
        global_step, train_log = 0, []
        torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
        started = time.perf_counter()
        for epoch in range(EPOCHS):
            model.train(); running = 0.0
            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")
            for step, batch in enumerate(progress_bar, start=1):
                group_start = ((step-1)//GRAD_ACCUM)*GRAD_ACCUM
                group_size = min(GRAD_ACCUM, len(train_loader)-group_start)
                batch = to_device(batch)
                with torch.autocast("cuda", dtype=DTYPE):
                    raw_loss = model(**batch, use_cache=False).loss
                if not torch.isfinite(raw_loss): raise FloatingPointError("NaN/Inf loss")
                (raw_loss/group_size).backward()
                running += float(raw_loss.detach())
                if step % GRAD_ACCUM == 0 or step == len(train_loader):
                    grads = [p.grad for p in params if p.grad is not None]
                    if not grads or not all(bool(torch.isfinite(g).all()) for g in grads):
                        raise FloatingPointError("LoRA gradient 누락 또는 NaN/Inf")
                    if global_step == 0 and not any(bool(g.abs().max()>0) for g in grads):
                        raise RuntimeError("LoRA gradient가 모두 0입니다.")
                    grad_norm = float(torch.nn.utils.clip_grad_norm_(params, 1.0))
                    optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
                    global_step += 1
                    train_log.append({"epoch":epoch+1, "update":global_step,
                        "loss":running/group_size, "grad_norm":grad_norm, "lr":scheduler.get_last_lr()[0]})
                    progress_bar.set_postfix(loss=f"{running/group_size:.4f}")
                    running = 0.0
                del batch, raw_loss
        torch.cuda.synchronize()
        training = {"state":"trained_here", "epochs":EPOCHS, "updates":global_step,
            "seconds":time.perf_counter()-started,
            "peak_allocated_gib":torch.cuda.max_memory_allocated()/2**30,
            "peak_reserved_gib":torch.cuda.max_memory_reserved()/2**30}
        pd.DataFrame(train_log).to_csv(COMMON_DIR / "train_log.csv", index=False)
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)
        base_model = model.unload()
        del model, optimizer, scheduler, params, grads
        gc.collect(); torch.cuda.empty_cache()
    write_json(COMMON_DIR / "training.json", training)
    # 완료 표시는 모든 파일 저장 후 기록합니다.
    tracked = [p for p in SAVE_DIR.rglob("*") if p.is_file()]
    tracked += [COMMON_DIR / "split_manifest.csv", COMMON_DIR / "image_audit.csv", COMMON_DIR / "training.json"]
    if (COMMON_DIR / "train_log.csv").exists(): tracked.append(COMMON_DIR / "train_log.csv")
    bundle = {"version":"resolution-v1", "model_id":MODEL_REPO, "model_revision":MODEL_REVISION,
        "train_csv_sha256":TRAIN_CSV_HASH, "packages":ENVIRONMENT["packages"], "model_files":files,
        "training_image_size":TRAIN_IMAGE_SIZE, "training_n":TRAIN_N,
        "files":{p.relative_to(COMMON_DIR).as_posix():file_hash(p) for p in tracked},
        "near_duplicate_review":"not_performed", "training":training}
    write_json(COMMON_DIR / "bundle.tmp.json", bundle)
    (COMMON_DIR / "bundle.tmp.json").replace(COMMON_DIR / "bundle.json")

model = PeftModel.from_pretrained(base_model, str(SAVE_DIR), local_files_only=True, is_trainable=False)
model.eval()
model.gradient_checkpointing_disable()
set_image_size(IMAGE_SIZE)
processor.save_pretrained(RUN_DIR / "processor")
CONFIG = {"experiment_id":EXPERIMENT_ID, "model_id":MODEL_REPO, "revision":MODEL_REVISION,
    "seed":SEED, "train_n":TRAIN_N, "valid_n":VALID_N, "holdout_n":HOLDOUT_N,
    "train_image_size":TRAIN_IMAGE_SIZE, "eval_image_size":IMAGE_SIZE,
    "min_pixels":IMAGE_SIZE**2, "max_pixels":IMAGE_SIZE**2,
    "epochs":EPOCHS, "batch_size":1, "gradient_accumulation":GRAD_ACCUM, "lr":LEARNING_RATE,
    "quantization":"NF4/double-quant/BF16-compute", "lora":{"r":8,"alpha":16,"dropout":0.05},
    "loss":"full_text; padding/media/role-special masked; EOS supervised",
    "max_new_tokens":MAX_NEW_TOKENS, "max_input_tokens":MAX_INPUT_TOKENS,
    "evaluation":"greedy2 + a-d constrained fallback; thinking disabled",
    "checkpoint_sha256":file_hash(SAVE_DIR / "adapter_model.safetensors"),
    "split_sha256":file_hash(COMMON_DIR / "split_manifest.csv"),
    "bundle_sha256":file_hash(COMMON_DIR / "bundle.json"),
    "train_csv_sha256":TRAIN_CSV_HASH, "test_csv_sha256":file_hash(DATA_DIR / "test.csv"),
    "sample_csv_sha256":file_hash(DATA_DIR / "sample_submission.csv"),
    "run_test":RUN_TEST, "dev_used":False, "holdout_evaluated":False, "kaggle_uploaded":False}
write_json(RUN_DIR / "run_config.json", CONFIG)
print("같은 해시인지 4대 결과에서 확인:", CONFIG["checkpoint_sha256"], CONFIG["split_sha256"])
print("PC2~4에 복사할 공통 폴더:", COMMON_DIR.resolve())


## 8. Inference·Accuracy
생성 토큰만 디코딩합니다. 파싱 실패는 기록하고 a~d 제한 생성으로 처리합니다.
검증에는 정답 메시지가 들어가지 않으며, 동일 해상도에서 원본과 저장 LoRA를 함께 평가합니다.


In [ ]:
def extract_choice(text):
    text = re.sub(r"^(?:answer|정답)\s*[:：]\s*", "", str(text).strip(), flags=re.I)
    match = re.fullmatch(r"(?:\(([a-d])\)|([a-d]))[.。]?", text, flags=re.I)
    return (match.group(1) or match.group(2)).lower() if match else None

choice_ids = [processor.tokenizer.encode(c, add_special_tokens=False) for c in "abcd"]
if not all(len(x)==1 for x in choice_ids) or len({x[0] for x in choice_ids}) != 4:
    raise ValueError("a~d가 단일·고유 토큰이 아닙니다.")
choice_ids = [x[0] for x in choice_ids]

def generate_choice(inputs, constrained=False):
    kwargs = dict(max_new_tokens=1 if constrained else MAX_NEW_TOKENS, do_sample=False,
                  num_beams=1, use_cache=True, repetition_penalty=1.0,
                  pad_token_id=processor.tokenizer.eos_token_id)
    if constrained:
        kwargs["prefix_allowed_tokens_fn"] = lambda batch_id, input_ids:choice_ids
    with torch.inference_mode(), torch.autocast("cuda", dtype=DTYPE):
        out_ids = model.generate(**inputs, **kwargs)
    new_ids = out_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new_ids, skip_special_tokens=True)[0].strip()

def evaluate(df, label, has_answers=True):
    # 입력 준비 전에 정답 컬럼을 제거합니다.
    clean_df = df.drop(columns=["answer"], errors="ignore")
    dataset = VQAMCDataset(clean_df, processor, train=False)
    loader = DataLoader(dataset, batch_size=1, shuffle=False,
        collate_fn=DataCollator(processor, False), num_workers=0)
    rows = []
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    started = time.perf_counter()
    fields = ["id","answer","raw_output","parse_failed","fallback_output","gold","correct","strict_correct","input_tokens","image_grid_thw"]
    with open(RUN_DIR / f"{label}_predictions.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields); writer.writeheader()
        for i, batch in enumerate(tqdm(loader, desc=label, unit="sample")):
            inputs = to_device(batch)
            raw = generate_choice(inputs)
            strict = extract_choice(raw)
            fallback = generate_choice(inputs, constrained=True) if strict is None else ""
            answer = strict if strict is not None else extract_choice(fallback)
            if answer is None: raise RuntimeError(f"ID {df.iloc[i]['id']}: 제한 생성 파싱 실패")
            gold = df.iloc[i]["answer"] if has_answers else ""
            row = {"id":df.iloc[i]["id"], "answer":answer, "raw_output":raw,
                "parse_failed":strict is None, "fallback_output":fallback, "gold":gold,
                "correct":answer==gold if has_answers else "",
                "strict_correct":strict==gold if has_answers else "",
                "input_tokens":int(inputs["input_ids"].shape[1]),
                "image_grid_thw":inputs["image_grid_thw"].tolist() if "image_grid_thw" in inputs else None}
            writer.writerow(row); f.flush(); rows.append(row)
            del inputs, batch
    torch.cuda.synchronize()
    seconds = time.perf_counter()-started
    metrics = {"n":len(rows), "seconds":seconds, "seconds_per_sample":seconds/len(rows),
        "peak_allocated_gib":torch.cuda.max_memory_allocated()/2**30,
        "peak_reserved_gib":torch.cuda.max_memory_reserved()/2**30,
        "parse_failure_rate":sum(r["parse_failed"] for r in rows)/len(rows),
        "max_input_tokens_observed":max(r["input_tokens"] for r in rows)}
    if has_answers:
        metrics.update(accuracy=sum(r["correct"] for r in rows)/len(rows),
                       strict_accuracy=sum(r["strict_correct"] for r in rows)/len(rows))
    write_json(RUN_DIR / f"{label}_metrics.json", metrics)
    print(label, metrics)
    return rows, metrics


In [ ]:
# 같은 이미지 해상도에서 원본과 저장된 LoRA를 비교합니다.
with model.disable_adapter():
    _, base_metrics = evaluate(valid_subset, "valid_base")
_, valid_metrics = evaluate(valid_subset, "valid_lora")


In [ ]:
# 베이스라인처럼 끝까지 실행하면 제출 CSV를 생성합니다.
test_metrics = None
if RUN_TEST:
    predictions, test_metrics = evaluate(test_df, "test", has_answers=False)
    pred_df = pd.DataFrame(predictions)[["id", "answer"]]
    if pred_df.id.duplicated().any() or set(pred_df.id) != set(test_df.id):
        raise ValueError("예측 ID 중복/누락")
    submission = sample_submission[["id"]].merge(pred_df, on="id", how="left", validate="one_to_one")
    submission = submission[list(sample_submission.columns)]
    if not submission.answer.isin(list("abcd")).all(): raise ValueError("유효하지 않은 예측값")
    if submission.id.tolist() != sample_submission.id.tolist(): raise ValueError("제출 ID 순서 불일치")
    SUBMISSION_PATH = RUN_DIR / "submission.csv"
    submission.to_csv(SUBMISSION_PATH, index=False)
    if not pd.read_csv(SUBMISSION_PATH, dtype=str, keep_default_na=False).equals(submission):
        raise ValueError("저장 후 제출 CSV 불일치")
    print("Saved:", SUBMISSION_PATH)
summary = {"config":CONFIG, "base":base_metrics, "lora":valid_metrics, "test":test_metrics,
    "base_outperformed_lora":base_metrics["accuracy"]>valid_metrics["accuracy"],
    "submission_checkpoint":"shared_lora", "public_score":None, "kaggle_uploaded":False,
    "status":"completed", "submission_created":RUN_TEST}
write_json(RUN_DIR / "summary.json", summary)


## 9. 결과 확인
4대의 `summary.json` / `run_config.json` / `valid_lora_predictions.csv`를 비교합니다.
체크포인트·분할·모델·환경은 같고 eval_image_size만 다른지 확인하세요.
로컬 Accuracy를 측정하며 Kaggle 제출·최고 모델 채택은 자동으로 수행하지 않습니다.


In [ ]:
print("이미지 픽셀 예산:", IMAGE_SIZE**2)
print("Base Accuracy:", base_metrics["accuracy"])
print("LoRA Accuracy:", valid_metrics["accuracy"])
print("LoRA parsing failure rate:", valid_metrics["parse_failure_rate"])
print("결과 폴더:", RUN_DIR.resolve())
if summary["base_outperformed_lora"]:
    print("이번 검증에서는 원본이 LoRA보다 높습니다. 제출 CSV는 지정한 공통 LoRA의 출력입니다.")
from IPython.display import display, FileLink
display(FileLink(str(RUN_DIR / "summary.json")))
if RUN_TEST: display(FileLink(str(SUBMISSION_PATH)))
